In [ ]:
# 1. Instalasi Library
!pip install -q streamlit google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 83.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

# Ambil Secret GEMINI
try:
    gemini_key = userdata.get('GEMINI')
    os.environ['GEMINI_API_KEY'] = gemini_key
    print("✅ Berhasil memuat API Key 'GEMINI' dari Colab Secrets!")
except Exception as e:
    print("❌ Gagal memuat 'GEMINI'. Pastikan secret 'GEMINI' sudah dibuat & diaktifkan.")

# Ambil Secret NGROK
try:
    ngrok_token = userdata.get('NGROK')
    os.environ['NGROK_AUTH_TOKEN'] = ngrok_token
    print("✅ Berhasil memuat Authtoken 'NGROK' dari Colab Secrets!")
except Exception as e:
    print("❌ Gagal memuat 'NGROK'. Pastikan secret 'NGROK' sudah dibuat & diaktifkan.")

✅ Berhasil memuat API Key 'GEMINI' dari Colab Secrets!
✅ Berhasil memuat Authtoken 'NGROK' dari Colab Secrets!


In [ ]:
with open("app.py", "w") as f:
    f.write('''import os
import streamlit as st
from google import genai
from google.genai import types

# --- KONFIGURASI HALAMAN ---
st.set_page_config(
    page_title="Nova — Assistant",
    page_icon="✨",
    layout="centered"
)

# --- STYLING CSS  ---
st.markdown("""
<style>
    /* Import Font Populer Gen Z */
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&display=swap');

    html, body, [class*="css"] {
        font-family: 'Plus Jakarta Sans', sans-serif;
    }

    /* Gradient Banner Modern Minimalis */
    .chat-header {
        background: linear-gradient(135deg, #a5b4fc 0%, #c084fc 50%, #f472b6 100%);
        padding: 20px 24px;
        border-radius: 20px;
        color: white;
        margin-bottom: 20px;
        box-shadow: 0 8px 20px rgba(192, 132, 252, 0.25);
        display: flex;
        align-items: center;
        gap: 15px;
    }
    .avatar-circle {
        width: 48px;
        height: 48px;
        background-color: rgba(255, 255, 255, 0.3);
        backdrop-filter: blur(8px);
        border-radius: 50%;
        display: flex;
        align-items: center;
        justify-content: center;
        font-size: 22px;
    }
    .header-text h2 {
        font-size: 22px;
        font-weight: 700;
        margin: 0;
        color: white !important;
    }
    .status-badge {
        font-size: 12px;
        background: rgba(255, 255, 255, 0.25);
        padding: 3px 10px;
        border-radius: 20px;
        font-weight: 500;
        display: inline-block;
        margin-top: 4px;
    }

    /* Sidebar Clean & Modern */
    [data-testid="stSidebar"] {
        background-color: #fafafa;
        border-right: 1px solid #f1f5f9;
    }

    /* Gelembung Chat User (Pill Shape Lavender) */
    [data-testid="stChatMessage"]:nth-child(even) {
        background-color: #e0e7ff;
        border-radius: 20px 20px 4px 20px;
        padding: 12px 18px;
        margin-bottom: 10px;
        color: #1e1b4b;
    }

    /* Gelembung Chat Nova (Soft Milk / Pastel) */
    [data-testid="stChatMessage"]:nth-child(odd) {
        background-color: #f3e8ff;
        border-radius: 20px 20px 20px 4px;
        padding: 12px 18px;
        margin-bottom: 10px;
        color: #3b0764;
    }

    /* Sembunyikan Elemen Khas AI dari Streamlit */
    [data-testid="stChatMessageAvatarUser"], [data-testid="stChatMessageAvatarAssistant"] {
        background: transparent !important;
    }

    /* Tombol Reset Style Modern Minimalis */
    div.stButton > button {
        background: #f43f5e;
        color: white;
        border-radius: 12px;
        border: none;
        padding: 10px 16px;
        font-weight: 600;
        width: 100%;
        transition: all 0.2s ease;
    }
    div.stButton > button:hover {
        background: #e11d48;
        transform: scale(0.98);
    }
</style>
""", unsafe_allow_html=True)

# --- HEADER TAMPILAN PERCAKAPAN ---
st.markdown("""
<div class="chat-header">
    <div class="avatar-circle">✨</div>
    <div class="header-text">
        <h2>Nova</h2>
        <div class="status-badge">🟢 Online — butuh apa?</div>
    </div>
</div>
""", unsafe_allow_html=True)

# --- AMBIL API KEY ---
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    st.error("Waduh, API Key 'GEMINI' belum ke-detect di Colab Secrets nih!")
    st.stop()

# --- SIDEBAR CONTROL ---
with st.sidebar:
    st.subheader("⚙️ Quick Settings")
    st.caption("Ngobrol santai, nanya tugas, bikin jadwal, atau sekadar nyari ide.")

    st.divider()
    if st.button("🧹 Clear Chat"):
        st.session_state.messages = []
        st.rerun()

# --- SYSTEM INSTRUCTION (CASUAL & NATURAL GEN Z VIBE) ---
system_instruction = """
Nama kamu adalah Nova. Kamu adalah teman/buddy harian yang santai, supportive, cerdas, dan asik diajak ngobrol (Gen Z friendly).

Gaya Bicara & Kepribadian:
- Gunakan bahasa sehari-hari, dan lu gue saja.
- Gunakan bahasa Indonesia santai yang natural, casual, dan akrab (pakai aku-kamu, sesekali gunakan istilah gaul/popular yang pas tanpa lebay).
- Hindari bahasa kaku seperti 'Tentu, berikut adalah...', 'Sebagai asisten AI...', atau penomoran yang terlalu tebal ala laporan kantor.
- Jawab dengan ringkas, to the point, dan langsung ke intinya. Pakai emoji secukupnya agar respon terasa hidup dan ramah.
- Kamu fleksibel: bisa diajak mikir ide kreatif, rapiin jadwal, buatin draf chat/caption, sampai bantu rangkum materi.
"""

# --- MEMORI CHAT ---
if "messages" not in st.session_state:
    st.session_state.messages = []

# Tampilkan riwayat chat tanpa ikon bot yang kaku
for message in st.session_state.messages:
    avatar = "💬" if message["role"] == "user" else "✨"
    with st.chat_message(message["role"], avatar=avatar):
        st.markdown(message["content"])

# --- PROSES CHAT ---
user_input = st.chat_input("Ketik sesuatu ke Aora...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user", avatar="💬"):
        st.markdown(user_input)

    client = genai.Client(api_key=api_key)

    contents = [
        types.Content(
            role="user" if m["role"] == "user" else "model",
            parts=[types.Part.from_text(text=m["content"])]
        ) for m in st.session_state.messages
    ]

    with st.chat_message("assistant", avatar="✨"):
        message_placeholder = st.empty()
        full_response = ""

        try:
            response = client.models.generate_content_stream(
                model="gemini-3.5-flash",
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.8,
                )
            )

            for chunk in response:
                full_response += chunk.text
                message_placeholder.markdown(full_response + "▌")

            message_placeholder.markdown(full_response)
            st.session_state.messages.append({"role": "assistant", "content": full_response})

        except Exception as e:
            st.error(f"Aduh, ada yang error nih: {e}")
            # --- FOOTER ---
st.markdown("""
<div class="app-footer">
    Nova Assistant by <span>Kevin</span>
</div>
""", unsafe_allow_html=True)
''')

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

# Ambil token dari environment variable
ngrok_auth_token = os.environ.get("NGROK_AUTH_TOKEN")

if not ngrok_auth_token:
    print("Authtoken NGROK tidak ditemukan di Secrets. Jalankan ulang Cell 2 terlebih dahulu.")
else:
    # Authenticate Ngrok
    ngrok.set_auth_token(ngrok_auth_token)

    # Jalankan Streamlit di background
    subprocess.Popen(["streamlit", "run", "app.py"])
    time.sleep(3)

    # Buka Tunnel ke port 8501
    public_url = ngrok.connect(8501)

    print("\n==============================================")
    print(f"🔗 LINK CHATBOT: {public_url}")
    print("==============================================")


🔗 LINK CHATBOT ANDA: NgrokTunnel: "https://student-slapstick-untainted.ngrok-free.dev" -> "http://localhost:8501"
